<a href="https://colab.research.google.com/github/YassinZohair/Real_or_Rendered/blob/main/notebooks/baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real or Rendered? Baseline (UnivFD)

Runs on a Colab GPU runtime. Cached features and results are written to `MyDrive/aml_real_or_rendered/`.

In [1]:
# Training size. 6 shards = Milestone 1 (10,505 images); 26 shards = about 44,000 images.
# Results and checkpoints for other sizes go to results_<n>shards/ and checkpoints_<n>shards/ on Drive.
%env ROR_TRAIN_SHARDS=6

env: ROR_TRAIN_SHARDS=6


In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; df -h /content | tail -1; nproc; python --version

Tesla T4, 15360 MiB
overlay         113G   83M  113G   1% /
2
Python 3.13.16


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
%cd /content
!test -d UniversalFakeDetect || git clone -q https://github.com/WisconsinAIVision/UniversalFakeDetect
!if test -d Real_or_Rendered; then git -C Real_or_Rendered pull -q; else git clone -q https://github.com/YassinZohair/Real_or_Rendered; fi
%cd /content/Real_or_Rendered
!pip install -q -r requirements.txt
!git log --oneline | head -3

/content
/content/Real_or_Rendered
37f7dad u3: noise branch (SRM + constrained conv + small ResNet) fused with CLIP; full model
31bb3d6 u1: B+U1+U2 starts from the U2 v2 head, so U1 is its only change
bd1f5f0 Created using Colab


## 1. Authors' code, weights and test data, unmodified

In [ ]:
!bash scripts/authors_eval.sh

Downloading...
From (original): https://drive.google.com/uc?id=1FXlGIRh_Ud3cScMgSVDbEWmPDmjcrm1t
From (redirected): https://drive.google.com/uc?id=1FXlGIRh_Ud3cScMgSVDbEWmPDmjcrm1t&confirm=t&uuid=a270869f-ddf8-464d-9b82-b373605b7087
To: /content/UniversalFakeDetect/diffusion_datasets.zip
100% 918M/918M [00:10<00:00, 86.6MB/s]
/content/UniversalFakeDetect/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
/content/UniversalFakeDetect/validate.py:21: DeprecationWarning: Please import `gaussian_filter` from the `scipy.ndimage` namespace; the `scipy.ndimage.filters` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.filters import gaussian_filter
100%|███████████████████████████████████████| 890M/890M [00:09<00:0

## 2. GenImage: download, cache CLIP features

In [ ]:
!python -m ror.data

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 62 files:   0% 0/62 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/1.80G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   1% 52.9M/3.83G [00:03<04:06, 15.3MB/s]
Reconstructing (incomplete total...):   2% 58.1M/3.83G [00:05<06:44, 9.33MB/s, 15.3MB/s  ]
Reconstructing (incomplete total...):   2% 65.5M/3.83G [00:05<05:52

In [ ]:
!python -m ror.features baseline

/content/UniversalFakeDetect/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
100%|███████████████████████████████████████| 890M/890M [00:22<00:00, 41.1MiB/s]
train_stable_diffusion_v_1_4.npz: cached, skipping
test_ADM.npz: cached, skipping
test_BigGAN.npz: cached, skipping
test_glide.npz: cached, skipping
test_Midjourney.npz: cached, skipping
test_stable_diffusion_v_1_4.npz: cached, skipping
test_stable_diffusion_v_1_5.npz: cached, skipping
test_VQDM.npz: cached, skipping
test_wukong.npz: cached, skipping


## 3. Check the cached pipeline against the repo's model

In [ ]:
!python -m ror.check_faithful

## 4. Baseline results per generator

In [ ]:
!python -m ror.baseline

In [ ]:
!python -m ror.plots baseline
from IPython.display import Image, display
from ror.plots import FIGURES
for path in sorted(FIGURES.glob('baseline_*.png')):
    display(Image(str(path), width=900))

## 5. U2: equal file format + real-world damage

Needs the GenImage files from step 2 (`python -m ror.data`) on this runtime.

In [ ]:
!python -m ror.features u2

In [ ]:
!python -m ror.u2

In [ ]:
!python -m ror.plots u2
from IPython.display import Image, display
from ror.plots import FIGURES
for path in sorted(FIGURES.glob('u2_*.png')):
    display(Image(str(path), width=900))

## 6. U1: LoRA on CLIP's attention

Low-rank adapters on the query and value projections of all 24 CLIP layers, trained end to end with the head.
Needs the GenImage files from step 2 and the baseline heads from step 4, at the same `ROR_TRAIN_SHARDS`.
If the session drops, re-run the same cell: training resumes from its last validation, and finished runs are not retrained.

In [ ]:
# Zero-init check: the adapted model must reproduce the cached baseline features, and only the adapters get gradients.
!python -m ror.u1 check

/content/UniversalFakeDetect/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
zero-init: max |feature difference| 0.00781, max |probability difference| 0.00015, same decision 100.0%
rank 8: 786,432 adapter parameters = 0.26% of CLIP's vision encoder (304,752,640); trainable tensors: 98 (adapters + head)
gradients: B matrices with a gradient in 24/24 layers, frozen tensors with a gradient: 0


In [ ]:
# Rank sweep: one seed, clean test images only.
for rank in [4, 8, 16]:
    !python -m ror.u1 train --rank {rank} --seed 0

/content/UniversalFakeDetect/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
step 0 (= baseline): val acc 0.9305
u1_r4 seed 0: 9455 images, 295 steps of 32, validating every 73
epoch 1:  24% 72/295 [01:35<05:12,  1.40s/it]step 73: train loss 0.2310, val acc 0.9486
epoch 1:  49% 145/295 [03:34<03:31,  1.41s/it]step 146: train loss 0.1603, val acc 0.9695
epoch 1:  74% 218/295 [05:31<01:47,  1.40s/it]step 219: train loss 0.1274, val acc 0.9752
epoch 1:  99% 291/295 [07:29<00:05,  1.40s/it]step 292: train loss 0.1232, val acc 0.9781
epoch 1: 100% 294/295 [07:48<00:03,  3.60s/it]step 295: train loss 0.1951, val acc 0.9781
epoch 1: 100% 295/295 [08:05<00:00,  1.65s/it]
kept step 292 of 295 (val acc 0.9781)
test_ADM_u1_r4_seed0.npz: 4000 imag

In [ ]:
!python -m ror.u1 report

                                                 acc  real_acc  fake_acc    ap   auc
protocol model          generator                                                   
resize   u1_r16         ADM                     51.7      97.1       6.4  58.0  57.7
                        BigGAN                  58.0      98.2      17.8  82.0  84.3
                        glide                   60.5      97.8      23.2  82.9  83.5
                        Midjourney              82.2      97.6      66.9  95.5  95.3
                        stable_diffusion_v_1_4  98.6      98.2      99.0  99.9  99.9
                        stable_diffusion_v_1_5  98.1      97.5      98.6  99.8  99.8
                        VQDM                    52.8      98.6       6.9  71.3  73.5
                        wukong                  95.2      98.2      92.2  99.3  99.2
         u1_r4          ADM                     51.5      95.9       7.2  56.0  56.0
                        BigGAN                  57.5      96.6   

In [ ]:
# Best rank from the sweep, 3 seeds, also tested on the damaged copies (seed 0 is not retrained).
BEST_RANK = 8
for seed in [0, 1, 2]:
    !python -m ror.u1 train --rank {BEST_RANK} --seed {seed} --damaged

/content/UniversalFakeDetect/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
test_ADM_u1_r8_seed0.npz: cached, skipping
test_BigGAN_u1_r8_seed0.npz: cached, skipping
test_glide_u1_r8_seed0.npz: cached, skipping
test_Midjourney_u1_r8_seed0.npz: cached, skipping
test_stable_diffusion_v_1_4_u1_r8_seed0.npz: cached, skipping
test_stable_diffusion_v_1_5_u1_r8_seed0.npz: cached, skipping
test_VQDM_u1_r8_seed0.npz: cached, skipping
test_wukong_u1_r8_seed0.npz: cached, skipping
                                        acc  real_acc  fake_acc    ap   auc
protocol model generator                                                   
resize   u1_r8 ADM                     51.9      96.9       7.0  57.8  57.2
               BigGAN                  57.

In [ ]:
# B+U1+U2: the same adapters trained on U2 v2 damage with equal file format, starting from the U2 v2 head.
BEST_RANK = 8  # chosen by the rank sweep
for seed in [0, 1, 2]:
    !python -m ror.u1 train --rank {BEST_RANK} --seed {seed} --u2 --damaged

## 7. U3: noise branch fused with CLIP

Fixed SRM and constrained high-pass filters, then a small CNN trained from scratch on 128 px crops at the image's own resolution.
CLIP stays frozen. The noise head starts at zero, so B+U3 starts exactly as B. Two fusions: concat and a learned gate.
Needs the GenImage files (step 2). The crops are cached on this runtime's disk (several GB), so rebuild them in a new session.

In [ ]:
!python -m ror.patches

In [ ]:
for fusion in ['concat', 'gate']:
    for seed in [0, 1, 2]:
        !python -m ror.u3 train --fusion {fusion} --seed {seed}

## 8. Full model: U1 + U2 + U3

U3 trained on top of each seed's B+U1+U2 model from step 6 (its adapted CLIP is frozen at this stage).

In [ ]:
!python -m ror.patches full

In [ ]:
for fusion in ['concat', 'gate']:
    for seed in [0, 1, 2]:
        !python -m ror.u3 train --fusion {fusion} --seed {seed} --full

In [ ]:
# Every model, mean over the 8 generators, clean and damaged.
!python -m ror.u3 report

### Or: every Milestone 2 run in one cell

The B+U1+U2, U3 and full runs in order. Each step skips finished work, so after a disconnect run it again.

In [6]:
!git pull -q && git log --oneline -1 && bash scripts/m2_runs.sh

a4df5c8 (HEAD -> main, origin/main, origin/HEAD) m2_runs: B+U3 first, so problems show up before the long B+U1+U2 runs
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 62 files:   0% 0/62 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/1.02G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/3.01G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.83G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   1% 44.8M/3.83G [00:03<06:01, 10.5MB/s, 17.9MB/s  ]
Reconstructing (incomplete total...):   1% 52.8M/3.83G [00:03<04:41, 13.4MB/s, 10.5MB/s  ]
Reconstructing (incomplete total...):   2% 61.6M/3.83G [00:04<03:33, 17.7M

In [7]:
!cd /content/Real_or_Rendered && git pull -q && git log --oneline -1 && python -m ror.analysis && python -m ror.plots m2 && cd /content/drive/MyDrive/aml_real_or_rendered && cp -r results m2_snapshot_2026-11-10_results && cp -r checkpoints m2_snapshot_2026-11-10_checkpoints && zip -qr m2_results.zip results && ls && du -sh m2_results.zip m2_snapshot_2026-11-10_*

3b2319d (HEAD -> main, origin/main, origin/HEAD) analysis and M2 figures: paired tests vs B, threshold diagnostic, every-model plots
resize  B+U2: done
resize  B+U1: done
resize  B+U1+U2: done
resize  B+U3: done
resize  Full (U1+U2+U3): done
jpeg95  B+U2: done
jpeg95  B+U1: done
jpeg95  B+U1+U2: done
jpeg95  B+U3: done
jpeg95  Full (U1+U2+U3): done
jpeg75  B+U2: done
jpeg75  B+U1: done
jpeg75  B+U1+U2: done
jpeg75  B+U3: done
jpeg75  Full (U1+U2+U3): done
jpeg50  B+U2: done
jpeg50  B+U1: done
jpeg50  B+U1+U2: done
jpeg50  B+U3: done
jpeg50  Full (U1+U2+U3): done
jpeg30  B+U2: done
jpeg30  B+U1: done
jpeg30  B+U1+U2: done
jpeg30  B+U3: done
jpeg30  Full (U1+U2+U3): done
shared  B+U2: done
shared  B+U1: done
shared  B+U1+U2: done
shared  B+U3: done
shared  Full (U1+U2+U3): done

Each model minus B, seed-averaged predictions, 95% bootstrap CIs (percentage points):
          model protocol generators  acc_diff  acc_diff_lo  acc_diff_hi  auc_diff  auc_diff_lo  auc_diff_hi  mcnemar_p
       

In [8]:
import pandas as pd
s = pd.read_csv('/content/drive/MyDrive/aml_real_or_rendered/results/analysis_significance.csv')
cols = ['model', 'protocol', 'generators', 'acc_diff', 'acc_diff_lo', 'acc_diff_hi', 'auc_diff', 'auc_diff_lo', 'auc_diff_hi', 'mcnemar_p']
print(s[cols].round(2).to_string(index=False))

          model protocol generators  acc_diff  acc_diff_lo  acc_diff_hi  auc_diff  auc_diff_lo  auc_diff_hi  mcnemar_p
           B+U2   resize      all 8      0.91         0.66         1.13      0.26         0.17         0.35       0.00
           B+U2   resize   7 unseen      1.13         0.87         1.39      0.30         0.19         0.40       0.00
           B+U1   resize      all 8     -0.59        -0.93        -0.23      3.51         3.30         3.72       0.00
           B+U1   resize   7 unseen     -1.42        -1.79        -1.03      3.80         3.56         4.04       0.00
        B+U1+U2   resize      all 8      1.72         1.40         2.05      4.39         4.15         4.60       0.00
        B+U1+U2   resize   7 unseen      1.26         0.92         1.64      4.81         4.54         5.06       0.00
           B+U3   resize      all 8      7.26         6.91         7.61     11.24        10.93        11.55       0.00
           B+U3   resize   7 unseen      7.52   

## 9. Score your own images

The whole pipeline on single images, no cache: test view, frozen CLIP, a trained head, P(AI).
Needs only cells 2 and 3 (Drive for the head, the repos for the code).

In [ ]:
from google.colab import files
uploaded = files.upload()
names = ' '.join(f'"{n}"' for n in uploaded)
# Any head in checkpoints/ works, e.g. u2v2_seed0.pth for the U2 model, or a U1 checkpoint such as u1_r8_seed0.pt.
!python -m ror.predict {names} --head /content/drive/MyDrive/aml_real_or_rendered/checkpoints/retrained_sd14_seed0.pth